##Seção 1 (Markdown) — Cabeçalho do Desafio

Você é um engenheiro em IA, e uma grande empresa lhe chamou para fazer um **Sistema de Recomendação** para eles. A recomendação de conteúdo está muito conectada a nós, quando assistimos **Netflix**, **YouTube**, **TikTok**, todo vídeo que é recomendado a nós é feito por um processo chamado "Sistema de Recomendação".
Com isso, precisamos, assim como Netflix, YouTube e TikTok, criar um sistema que adivinhe o que vamos gostar, sua missão é treinar um modelo que sugira o próximo filme perfeito.

Mas calma, Não precisa maratonar código: as engrenagens já existem. Seu papel é tomar boas decisões para guiá-las.
Para montar esse sistema precisamos de 3 peças:
Dados — filmes, usuários e notas que eles deram;
Preparação — polir esses dados para que a IA enxergue melhor;
Modelo — o cérebro que aprende com tudo isso e faz as sugestões.

##Regras
- ⚠️ Células marcadas com **"não altere esta célula"** devem ser mantidas intactas.
- 💻 Células marcadas com **"SEU CÓDIGO AQUI"** são onde você deve implementar sua solução.
- 🛑 Não altere o nome do arquivo de saída (`submission.csv`) e não utilize `google.colab` para download.

[https://i.ibb.co/v4m6pM8f/cinema-recommendation.png](https://i.ibb.co/v4m6pM8f/cinema-recommendation.png)


##Seção 2 (Código) — Célula de Configuração

In [ ]:
# ============================================================
# CONFIGURAÇÃO — não altere esta célula
# ============================================================

# 1. Instalação de dependências
import subprocess
import sys

try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    import gdown

# 2. Imports padrão
import os
import warnings
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import accuracy_score

warnings.filterwarnings('ignore')

# 3. Seed para reprodutibilidade
SEED = 42
np.random.seed(SEED)

# 4. Paths padrão (nunca hardcode fora desta célula)
DADOS_DIR = 'competition_data'
OUTPUT_FILE = 'predictions.csv'

print(f'Configuração OK | Seed: {SEED}')

##Seção 3 (Código) — Célula de Carregamento de Dados

In [ ]:
# ============================================================
# CARREGAMENTO DE DADOS — não altere esta célula
# ============================================================

def carregar_dados_competicao(pasta_destino: str = DADOS_DIR):
    """
    Baixa e carrega os dados da competição.
    Retorna: (competition_train, test_df_submissao, sample_submission)
    """
    print("Carregando dados da competição do Google Drive...")
    os.makedirs(pasta_destino, exist_ok=True)

    def get_direct_download_link(drive_url):
        file_id = drive_url.split('/d/')[1].split('/')[0]
        return f"https://drive.google.com/uc?id={file_id}"

    files_to_download = {
        'train.csv': 'https://drive.google.com/file/d/1oFx2twXvZD8HHjVGU8qM-SJ43cK2P0YS/view?usp=drive_link',
        'test.csv': 'https://drive.google.com/file/d/1KR4JCCVdROUbbGRyLq52sSrX6b2U6euE/view?usp=drive_link',
        'sample_submission.csv': 'https://drive.google.com/file/d/1zNEeejF4OeNJzNCn4f8nhLU1BWuYwl5L/view?usp=drive_link'
    }

    print("Baixando arquivos da competição...")
    for filename, url in files_to_download.items():
        direct_url = get_direct_download_link(url)
        print(f"Baixando {filename}...")
        gdown.download(direct_url, f'{pasta_destino}/{filename}', quiet=False)

    competition_train = pd.read_csv(os.path.join(pasta_destino, 'train.csv'))
    test_df_submissao = pd.read_csv(os.path.join(pasta_destino, 'test.csv'))
    sample_submission = pd.read_csv(os.path.join(pasta_destino, 'sample_submission.csv'))

    return competition_train, test_df_submissao, sample_submission

# Executar carregamento
competition_train, test_df_submissao, sample_submission = carregar_dados_competicao()

# Visualização inicial
print('\n--- Primeiras linhas do Treino ---')
display(competition_train.head())
print('\n--- Tipos e nulos do Treino ---')
display(competition_train.info())

##Seção 4 (Código) — Funções Auxiliares (Processamento e Modelagem)

In [ ]:
# ============================================================
# FUNÇÕES AUXILIARES — não altere esta célula
# ============================================================

def prepare_data_consistently(train_df, val_df, test_df, normalize_ids=False, normalize_ratings=True):
    """
    Normaliza os dados de forma consistente entre treino, validação e teste.
    """
    print("\n🔧 APLICANDO NORMALIZAÇÃO CONSISTENTE...")
    all_data = pd.concat([train_df, val_df, test_df], ignore_index=True)

    train_processed = train_df.copy()
    val_processed = val_df.copy()
    test_processed = test_df.copy()

    normalization_params = {}

    if normalize_ids:
        user_min, user_max = all_data['userId'].min(), all_data['userId'].max()
        movie_min, movie_max = all_data['movieId'].min(), all_data['movieId'].max()

        for df in [train_processed, val_processed, test_processed]:
            df['userId'] = (df['userId'] - user_min) / (user_max - user_min)
            df['movieId'] = (df['movieId'] - movie_min) / (movie_max - movie_min)

        normalization_params['user_min'] = user_min
        normalization_params['user_max'] = user_max
        normalization_params['movie_min'] = movie_min
        normalization_params['movie_max'] = movie_max

        print(f"✅ IDs normalizados: usuários [{user_min}-{user_max}] → [0-1]")
        print(f"✅ IDs normalizados: filmes [{movie_min}-{movie_max}] → [0-1]")

    if normalize_ratings:
        rating_min = train_df['rating'].min()
        rating_max = train_df['rating'].max()

        train_processed['rating'] = (train_processed['rating'] - rating_min) / (rating_max - rating_min)
        val_processed['rating'] = (val_processed['rating'] - rating_min) / (rating_max - rating_min)

        normalization_params['rating_min'] = rating_min
        normalization_params['rating_max'] = rating_max

        print(f"✅ Ratings normalizados: [{rating_min}-{rating_max}] → [0-1]")

    if not normalize_ids and not normalize_ratings:
        print("✅ Nenhuma normalização aplicada")

    return train_processed, val_processed, test_processed, normalization_params


def evaluate_model_corrected(model, df_eval, norm_params, threshold=3.5):
    """
    Avalia o desempenho do modelo utilizando RMSE e Hit Rate@5.
    """
    is_normalized = 'rating_min' in norm_params

    if is_normalized:
        rating_min = norm_params['rating_min']
        rating_max = norm_params['rating_max']
        threshold_norm = (threshold - rating_min) / (rating_max - rating_min)
        threshold_norm = np.clip(threshold_norm, 0, 1)
        print(f"✅ Ratings normalizados detectados")
        print(f"✅ Threshold ajustado: {threshold} → {threshold_norm:.2f}")
        actual_threshold = threshold_norm
    else:
        print(f"✅ Ratings originais, threshold: {threshold}")
        actual_threshold = threshold

    predictions = []
    actuals = []
    hits = 0
    total_users = 0

    user_ratings = df_eval.groupby('userId')

    for user_id, user_df in user_ratings:
        if user_id not in model.user_mapper:
            continue

        user_preds = []
        for _, row in user_df.iterrows():
            if row['movieId'] in model.item_mapper:
                pred = model.predict(user_id, row['movieId'])
                user_preds.append((row['movieId'], pred))
                predictions.append(pred)
                actuals.append(row['rating'])

        if not user_preds:
            continue

        liked_items = set(user_df[user_df['rating'] >= actual_threshold]['movieId'])
        if liked_items:
            user_preds.sort(key=lambda x: x[1], reverse=True)
            top_5 = set(item for item, _ in user_preds[:5])
            if top_5.intersection(liked_items):
                hits += 1

        total_users += 1

    rmse = np.sqrt(mean_squared_error(actuals, predictions)) if actuals else 0
    hit_rate = (hits / total_users * 100) if total_users > 0 else 0

    print(f"📊 RMSE: {rmse:.4f}")
    print(f"🎯 Hit Rate@5: {hit_rate:.2f}%\")")
    print(f"👥 Avaliados {total_users} usuários, {len(actuals)} predições")

    if predictions:
        print(f"🔍 Predições: min={min(predictions):.3f}, max={max(predictions):.3f}, média={np.mean(predictions):.3f}")

    return rmse, hit_rate


def generate_submission_corrected(model, test_df, sample_submission, norm_params):
    """
    Gera o arquivo de predições estruturado para submissão.
    """
    print("\n🔧 GERANDO SUBMISSÃO CORRIGIDA...")
    submission_predictions = []

    for idx, row in sample_submission.iterrows():
        test_row = test_df.iloc[idx]
        user_id = test_row['userId']
        movie_id = test_row['movieId']

        pred = model.predict(user_id, movie_id)
        submission_predictions.append(pred)

    predictions_processed = process_predictions_corrected(submission_predictions, norm_params)

    submission = pd.DataFrame({
        'id': sample_submission['id'],
        'rating': predictions_processed
    })

    return submission


def process_predictions_corrected(predictions, norm_params):
    predictions = np.array(predictions)

    if 'rating_min' in norm_params and 'rating_max' in norm_params:
        rating_min = norm_params['rating_min']
        rating_max = norm_params['rating_max']
        predictions = np.clip(predictions, 0.0, 1.0)

        print(f"🔄 Desnormalizando: [0-1] → [{rating_min}-{rating_max}]")
        predictions = predictions * (rating_max - rating_min) + rating_min
        print(f"📊 Após desnormalização: min={predictions.min():.3f}, max={predictions.max():.3f}")
    else:
        print("ℹ️ Nenhuma normalização aplicada — mantendo predições originais.")

    predictions = np.round(predictions).astype(int)
    predictions = np.clip(predictions, 1, 5)

    return predictions.tolist()

##Seção 5 (Markdown + Código) — Modelos Disponíveis

In [ ]:
# ============================================================
# MODELOS DISPONÍVEIS — não altere esta célula
# ============================================================

class RecommenderBase:
    def __init__(self, df):
        self.df = df
        self.user_movie_matrix = None
        self.user_mapper = {}
        self.item_mapper = {}
        self.user_inv_mapper = {}
        self.item_inv_mapper = {}
        self.global_mean = 0

    def prepare_data(self):
        users = self.df['userId'].unique()
        items = self.df['movieId'].unique()

        self.user_mapper = {user_id: idx for idx, user_id in enumerate(users)}
        self.item_mapper = {item_id: idx for idx, item_id in enumerate(items)}
        self.user_inv_mapper = {v: k for k, v in self.user_mapper.items()}
        self.item_inv_mapper = {v: k for k, v in self.item_mapper.items()}

        user_index = self.df['userId'].map(self.user_mapper)
        item_index = self.df['movieId'].map(self.item_mapper)

        self.user_movie_matrix = np.zeros((len(users), len(items)))
        for u, i, r in zip(user_index, item_index, self.df['rating']):
            self.user_movie_matrix[u, i] = r

        self.global_mean = self.user_movie_matrix[self.user_movie_matrix != 0].mean()


class FunkSVDModel(RecommenderBase):
    def train(self, n_factors=15, n_epochs=50, lr=0.01, reg=0.02):
        self.prepare_data()
        n_users, n_items = self.user_movie_matrix.shape

        scale = np.sqrt(self.global_mean / n_factors)
        self.P = np.random.normal(0, scale, size=(n_users, n_factors))
        self.Q = np.random.normal(0, scale, size=(n_items, n_factors))

        self.user_bias = np.zeros(n_users)
        self.item_bias = np.zeros(n_items)

        for user_id, user_idx in self.user_mapper.items():
            user_ratings = self.df[self.df['userId'] == user_id]['rating']
            if len(user_ratings) > 0:
                self.user_bias[user_idx] = user_ratings.mean() - self.global_mean

        for item_id, item_idx in self.item_mapper.items():
            item_ratings = self.df[self.df['movieId'] == item_id]['rating']
            if len(item_ratings) > 0:
                self.item_bias[item_idx] = item_ratings.mean() - self.global_mean

        initial_lr = lr
        train_coords = list(zip(*self.user_movie_matrix.nonzero()))

        for epoch in range(n_epochs):
            current_lr = initial_lr * (0.98 ** epoch)
            np.random.shuffle(train_coords)
            total_loss = 0

            for user_idx, item_idx in train_coords:
                rating = self.user_movie_matrix[user_idx, item_idx]
                pred = (self.global_mean + self.user_bias[user_idx] + self.item_bias[item_idx] + self.P[user_idx].dot(self.Q[item_idx]))
                error = rating - pred
                total_loss += error ** 2

                p_old = self.P[user_idx].copy()
                self.P[user_idx] += current_lr * (error * self.Q[item_idx] - reg * self.P[user_idx])
                self.Q[item_idx] += current_lr * (error * p_old - reg * self.Q[item_idx])
                self.user_bias[user_idx] += current_lr * (error - reg * self.user_bias[user_idx])
                self.item_bias[item_idx] += current_lr * (error - reg * self.item_bias[item_idx])

            if epoch % 10 == 0:
                rmse = np.sqrt(total_loss / len(train_coords))
                print(f"Época {epoch}: RMSE = {rmse:.4f}")

        print("✅ Treinamento concluído!")

    def predict(self, user_id, item_id):
        if user_id not in self.user_mapper or item_id not in self.item_mapper:
            return self.global_mean
        user_idx = self.user_mapper[user_id]
        item_idx = self.item_mapper[item_id]
        pred = (self.global_mean + self.user_bias[user_idx] + self.item_bias[item_idx] + self.P[user_idx].dot(self.Q[item_idx]))
        return float(np.clip(pred, 0.0, 1.0)) if hasattr(self, "norm_params") and "rating_min" in self.norm_params else float(np.clip(pred, 1.0, 5.0))


class MFModel(RecommenderBase):
    def train(self, n_factors=20, n_epochs=50, lr=0.015, reg=0.15):
        self.prepare_data()
        n_users, n_items = self.user_movie_matrix.shape

        scale = np.sqrt(self.global_mean / n_factors)
        self.P = np.random.normal(0, scale, size=(n_users, n_factors))
        self.Q = np.random.normal(0, scale, size=(n_items, n_factors))

        self.user_bias = np.zeros(n_users)
        self.item_bias = np.zeros(n_items)
        train_coords = list(zip(*self.user_movie_matrix.nonzero()))

        for epoch in range(n_epochs):
            np.random.shuffle(train_coords)
            for user_idx, item_idx in train_coords:
                rating = self.user_movie_matrix[user_idx, item_idx]
                pred = (self.global_mean + self.user_bias[user_idx] + self.item_bias[item_idx] + self.P[user_idx].dot(self.Q[item_idx]))
                error = rating - pred

                p_old = self.P[user_idx].copy()
                self.P[user_idx] += lr * (error * self.Q[item_idx] - reg * self.P[user_idx])
                self.Q[item_idx] += lr * (error * p_old - reg * self.Q[item_idx])
                self.user_bias[user_idx] += lr * (error - reg * self.user_bias[user_idx])
                self.item_bias[item_idx] += lr * (error - reg * self.item_bias[item_idx])

    def predict(self, user_id, item_id):
        if user_id not in self.user_mapper or item_id not in self.item_mapper:
            return self.global_mean
        user_idx = self.user_mapper[user_id]
        item_idx = self.item_mapper[item_id]
        pred = (self.global_mean + self.user_bias[user_idx] + self.item_bias[item_idx] + self.P[user_idx].dot(self.Q[item_idx]))
        return float(np.clip(pred, 0.0, 1.0)) if hasattr(self, "norm_params") and "rating_min" in self.norm_params else float(np.clip(pred, 1.0, 5.0))


class BaselineModel(RecommenderBase):
    def train(self, damping_factor=10):
        self.prepare_data()
        self.user_bias = {}
        self.item_bias = {}

        for user_id in self.df['userId'].unique():
            user_ratings = self.df[self.df['userId'] == user_id]['rating']
            user_bias = (user_ratings - self.global_mean).sum() / (len(user_ratings) + damping_factor)
            self.user_bias[user_id] = user_bias

        for item_id in self.df['movieId'].unique():
            item_ratings = self.df[self.df['movieId'] == item_id]['rating']
            item_bias = (item_ratings - self.global_mean).sum() / (len(item_ratings) + damping_factor)
            self.item_bias[item_id] = item_bias

    def predict(self, user_id, item_id):
        pred = self.global_mean
        if user_id in self.user_bias:
            pred += self.user_bias[user_id]
        if item_id in self.item_bias:
            pred += self.item_bias[item_id]
        return float(np.clip(pred, 0.0, 1.0)) if hasattr(self, "norm_params") and "rating_min" in self.norm_params else float(np.clip(pred, 1.0, 5.0))

##Seção 6 (Código) — Pipeline do Aluno (TODO)

In [ ]:
# ============================================================
# PIPELINE DO ALUNO (TODO)
# ============================================================

# PASSO 1: Escolha o tamanho da divisão dos dados (Ex: 80 para 80% treino e 20% validação)
train_size_percentage = 80
train_df_raw, val_df_raw = train_test_split(competition_train, train_size=train_size_percentage/100, random_state=42)

# PASSO 2: Defina as configurações de normalização desejadas
normalize_ids = False
normalize_ratings = True

train_df, val_df, test_df_processed, norm_params = prepare_data_consistently(
    train_df_raw, val_df_raw, test_df_submissao, normalize_ids, normalize_ratings
)

# PASSO 3: Selecione o modelo descomentando a linha correspondente
# Opções: FunkSVDModel(train_df), MFModel(train_df), BaselineModel(train_df)
model = FunkSVDModel(train_df)
model.norm_params = norm_params

# PASSO 4: Defina os hiperparâmetros e execute o treinamento
epochs = 50
factors = 15

if isinstance(model, BaselineModel):
    model.train()
else:
    model.train(n_factors=factors, n_epochs=epochs)

##Seção 7 (Código) — Avaliação

In [ ]:
# ============================================================
# AVALIAÇÃO — não altere
# ============================================================

print("\n🔍 --- AVALIAÇÃO DO MODELO ---")
rmse, hit_rate = evaluate_model_corrected(model, val_df, norm_params, threshold=3.5)

##Seção 8 (Código) — Geração do submission.csv

In [ ]:
# ============================================================
# EXPORTAÇÃO — não altere
# ============================================================

submission = generate_submission_corrected(model, test_df_processed, sample_submission, norm_params)
submission.to_csv(OUTPUT_FILE, index=False)

print(f'\n📁 Arquivo salvo: {OUTPUT_FILE}')
print(f'   Linhas: {len(submission)} | Colunas: {list(submission.columns)}')
display(submission.head())

# Download automático no ambiente Colab
from google.colab import files
files.download(OUTPUT_FILE)
print("Submissão baixada!")

##Seção 9 (Código) — Exploração Livre

# ============================================================
# EXPLORAÇÃO LIVRE — Células extras para o aluno (Opcional)
# ============================================================

# Use este espaço se quiser testar engenharia de atributos adicional,
# tunagem de hiperparâmetros ou novos algoritmos sem quebrar as células fixas acima.